In [5]:
import pandas as pd
import numpy as np

def compare_and_merge_csv(file1_path, file2_path, key_column, output_csv_path, log_file_path):
    """
    2つのCSVファイルを比較・結合し、指定されたロジックで出力ファイルとログファイルを生成する。

    Args:
        file1_path (str): 1つ目のCSVファイルパス
        file2_path (str): 2つ目のCSVファイルパス
        key_column (str): 結合のキーとなる列名（例：「品番」）
        output_csv_path (str): 結果を出力するCSVファイルパス
        log_file_path (str): 値の不一致を記録するログファイルパス
    """
    try:
        # CSVファイルをpandas DataFrameとして読み込む
        # index_colでキー列を指定することで、後の処理が容易になる
        df1 = pd.read_csv(file1_path, index_col=key_column)
        df2 = pd.read_csv(file2_path, index_col=key_column)
        print("CSVファイルの読み込みに成功しました。")
    except FileNotFoundError as e:
        print(f"エラー: ファイルが見つかりません。 {e}")
        return
    except Exception as e:
        print(f"ファイルの読み込み中にエラーが発生しました: {e}")
        return

    # ログファイルを開く準備
    with open(log_file_path, 'w', encoding='utf-8-sig') as log_file:
        log_file.write("品番,列名,ファイル1の値,ファイル2の値\n")
        print(f"ログファイルを '{log_file_path}' に作成しました。")

        # 最終的な結果を格納するDataFrameを準備（インデックスは両方のキーを統合したもの）
        all_keys = df1.index.union(df2.index)
        result_df = pd.DataFrame(index=all_keys)

        # 列名のセットを取得
        cols1 = set(df1.columns)
        cols2 = set(df2.columns)
        
        # 共通の列を処理
        common_cols = cols1.intersection(cols2)
        print(f"共通の列を処理中: {list(common_cols)}")
        for col in common_cols:
            # result_dfに新しい列を追加（object型で初期化）
            result_df[col] = pd.Series(dtype='object')
            # 品番ごとに値を比較
            for key in all_keys:
                # 両方のDataFrameにキーと列が存在するか確認
                val1_exists = key in df1.index
                val2_exists = key in df2.index
                
                val1 = df1.loc[key, col] if val1_exists else np.nan
                val2 = df2.loc[key, col] if val2_exists else np.nan

                # 値を比較
                # pd.isna()でNaNかどうかを判定
                if pd.isna(val1) and pd.isna(val2):
                    result_df.loc[key, col] = np.nan
                elif val1 == val2:
                    result_df.loc[key, col] = val1
                else:
                    # 値が異なる場合 (片方だけNaNの場合も含む)
                    result_df.loc[key, col] = np.nan
                    # 両方の値が存在し、かつ異なる場合のみログに記録
                    if val1_exists and val2_exists and val1 != val2:
                        log_file.write(f'"{key}","{col}","{val1}","{val2}"\n')
        
        # file1にしかない列を処理
        unique_cols1 = cols1.difference(cols2)
        if unique_cols1:
            print(f"ファイル1固有の列を追加中: {list(unique_cols1)}")
            # df1から該当列をresult_dfに結合（indexが自動で合う）
            result_df = result_df.join(df1[list(unique_cols1)])

        # file2にしかない列を処理
        unique_cols2 = cols2.difference(cols1)
        if unique_cols2:
            print(f"ファイル2固有の列を追加中: {list(unique_cols2)}")
            # df2から該当列をresult_dfに結合
            result_df = result_df.join(df2[list(unique_cols2)])

        # 結果をCSVファイルに保存
        # index=Trueで品番列も出力する
        result_df.to_csv(output_csv_path, index=True, encoding='utf-8-sig')
        print(f"処理が完了しました。結果を '{output_csv_path}' に保存しました。")


# --- ここから設定 ---
# 実行するには、以下のファイルパスを実際の環境に合わせて変更してください

# 入力ファイル
file1 = '../data/Data_20251014.csv'
file2 = '../data/DataData_NonWire_20251015_Rep付.csv'

# キーとなる列名
key = '品番'

# 出力ファイル
output_file = '../data/20251016_merged_output.csv'
log_file = '../data/Data_merge_conflict_log.csv'

# 関数を実行
compare_and_merge_csv(file1, file2, key, output_file, log_file)

CSVファイルの読み込みに成功しました。
ログファイルを '../data/Data_merge_conflict_log.csv' に作成しました。
共通の列を処理中: ['ステップ', 'シーズン', 'タイプ', 'tsne_x', 'シーン2', 'tsne_y', 'ブランド名', 'medoid_score', '補整力', '上代数値', 'シルエット', 'シーン1', 'チャネル', 'cluster']
ファイル1固有の列を追加中: ['ワイヤー']
ファイル2固有の列を追加中: ['Rep']
処理が完了しました。結果を '../data/20251016_merged_output.csv' に保存しました。
